# Rural Urban Classifications for the UK

Rural/Urban classifications are not consistent across the UK. We are using a simplified rural/urban flag here. Further research should look into the differences and similarities across the 4 regions to enable better comparisons.

In [ ]:
from pathlib import Path

BASE = Path('/Users/pedrickr/Documents/ADR/Connectivity')

PATH_LSOA = BASE / 'Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFC_V10_-7599572456947714539/LSOA_2021_EW_BFC_V10.shp'
PATH_RURB = BASE / 'Rural_Urban_Classification_(2021)_of_LSOAs_in_EW.csv'
PATH_SCOT_RURB = BASE / 'SG_UrbanRural_2022/SG_UrbanRural_2022.shp'
PATH_SCOT_OA = BASE / 'Scotland/outputarea2022_partremoved_mhw/OutputArea2022_PartRemoved_MHW/OutputArea2022_PartRemoved_MHW.shp'
PATH_NI_SA = BASE / 'NI/SA2011_Esri_Shapefile_0/SA2011.shp'
PATH_NI_RURB = BASE / 'NI/NI_SA_2011_UR.csv'

In [ ]:
import geopandas as gpd
import pandas as pd
import gc

# ---------------------
# England & Wales
# ---------------------

lsoa_2021 = gpd.read_file(PATH_LSOA)[['LSOA21CD', 'geometry']]

lsoa_rurb_2021 = pd.read_csv(PATH_RURB)[
    ['LSOA21CD', 'Urban_rural_flag']
]

lsoa_rurb_poly = lsoa_2021.merge(
    lsoa_rurb_2021,
    on='LSOA21CD',
    how='left'
)

rural_lsoas = lsoa_rurb_poly[
    lsoa_rurb_poly['Urban_rural_flag'] == 'Rural'
]

lsoa_total = lsoa_2021['LSOA21CD'].nunique()
rural_lsoa_total = rural_lsoas['LSOA21CD'].nunique()

print(f"LSOAs total: {lsoa_total}")
print(f"Rural LSOAs: {rural_lsoa_total}")
print(f"% rural: {round(rural_lsoa_total / lsoa_total * 100, 3)}")

del lsoa_rurb_poly
gc.collect()

# ---------------------
# Scotland
# ---------------------

scot_rurb = gpd.read_file(PATH_SCOT_RURB)[['UR2Name', 'geometry']]
scot_oa   = gpd.read_file(PATH_SCOT_OA)[['code', 'geometry']]

# Ensure CRS match
scot_oa = scot_oa.to_crs(scot_rurb.crs)

scot_rural = scot_rurb[
    scot_rurb['UR2Name'] == 'Rural Areas'
]

scot_oa_rural = gpd.sjoin(
    scot_oa,
    scot_rural,
    how='inner',
    predicate='intersects'
)

scot_oa_total = scot_oa['code'].nunique()
scot_oa_rural_total = scot_oa_rural['code'].nunique()

print(f"Scotland OA total: {scot_oa_total}")
print(f"Scotland rural OA: {scot_oa_rural_total}")
print(f"% rural: {round(scot_oa_rural_total / scot_oa_total * 100, 3)}")

del scot_oa_rural
gc.collect()

# ---------------------
# Northern Ireland
# ---------------------

ni_sa_2011 = gpd.read_file(PATH_NI_SA)[['SA2011', 'geometry']]
ni_sa_ru   = pd.read_csv(PATH_NI_RURB)

ni_rural_codes = set(
    ni_sa_ru.loc[
        ni_sa_ru['UrbanRural Status'] == 'RURAL',
        'SACode'
    ]
)

ni_sa_rural = ni_sa_2011[
    ni_sa_2011['SA2011'].isin(ni_rural_codes)
]

ni_total = ni_sa_2011['SA2011'].nunique()
ni_rural_total = ni_sa_rural['SA2011'].nunique()

print(f"NI total SAs: {ni_total}")
print(f"NI rural SAs: {ni_rural_total}")
print(f"% rural: {round(ni_rural_total / ni_total * 100, 3)}")

# ---------------------
# Combine UK geometries
# ---------------------

target_crs = lsoa_2021.crs

all_uk = gpd.GeoDataFrame(
    pd.concat([
        lsoa_2021[['geometry']],
        scot_rurb[['geometry']].to_crs(target_crs),
        ni_sa_2011[['geometry']].to_crs(target_crs)
    ], ignore_index=True),
    crs=target_crs
)

rural_uk = gpd.GeoDataFrame(
    pd.concat([
        rural_lsoas[['geometry']],
        scot_rural[['geometry']].to_crs(target_crs),
        ni_sa_rural[['geometry']].to_crs(target_crs)
    ], ignore_index=True),
    crs=target_crs
)

rural_uk['rural'] = 1

# Only dissolve if needed
rural_dissolved = rural_uk.dissolve()

gc.collect()

In [ ]:
# Plot

ax = all_uk.plot(
    edgecolor="black",
    linewidth=0.6,
    facecolor="none",
    figsize=(6, 8)
)


rural_dissolved.plot(
    ax=ax,
    color="darkkhaki",
    edgecolor="none"
)

ax.set_title("Rural Areas of the United Kingdom (NI 2015 (2011), Scotland 2022, England & Wales 2021)", fontsize=14)
ax.set_axis_off()

ax.text(
    0.5, 1.01,
    "Derived from census geographies",
    transform=ax.transAxes,
    ha="center",
    fontsize=10
)

ax.text(
    0.01, 0.01,
    "Source: ONS, Scottish Government, NISRA",
    transform=ax.transAxes,
    fontsize=8,
    va="bottom"
)

ax.figure.savefig("rural_uk.png", dpi=300, bbox_inches='tight')